# Template 07a: SHAP Summary

**Purpose:** Overall SHAP results, most important first:
1. Beeswarm by original column (one-hot levels summed per record) - top features at a glance
2. Key numbers (features used, coverage, train vs test agreement)
3. Importance by original column
4. Beeswarm and importance by model feature (one-hot levels shown separately)
5. Train vs test importance, cumulative importance

**Importance** = exposure-weighted mean of |SHAP| per record. For one-hot columns the levels are
summed within each record first (record level), then |.| is taken. Charts are ranked by **test**
importance (full test set); each beeswarm uses exactly the order of its bar chart. Beeswarms are
drawn from the train SHAP sample (dots drawn proportional to exposure).

**Inputs:** results/06_shap_{train,test}.parquet, results/05c_predictions_{train,test}.parquet,
data/04c_train_encoded.parquet, results/04c_encoding_summary.csv

**Outputs:**
- results/07a_shap_importance.csv (every model feature, train + test stats)
- results/07a_shap_importance_by_column.csv (every original column, record level, train + test stats)
- results/07a_column_shap_train.parquet (record-level SHAP per original column, train sample)
- results/07a_summary.yaml
- plots/07a_beeswarm_columns.png, 07a_importance_columns.png, 07a_beeswarm_features.png,
  07a_importance_features.png, 07a_train_vs_test.png, 07a_cumulative.png

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import os
import sys
import yaml
import pandas as pd
import matplotlib
matplotlib.use("Agg")
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment, save_parquet
project_root = setup_notebook_environment()
import shap_analysis as sa

print("########################################")
print("# STAGE 07a: SHAP SUMMARY")
print("########################################")

In [ ]:
pc_id = f"PC{open('current.pc').read().strip()}"
with open(f"{config_path}/config.yaml") as f:
    cfg = yaml.safe_load(f)
output_base = cfg["machines"][pc_id]["paths"]["output_path"]
shap_cfg = cfg.get("shap_analysis", {})
top_n = shap_cfg.get("top_n_features", 40)
beeswarm_rows = shap_cfg.get("beeswarm_rows", 50000)
results_dir = f"{output_base}/results"
plots_dir = f"{output_base}/plots"
os.makedirs(plots_dir, exist_ok=True)
print(f"Output: {output_base} | top_n={top_n} | beeswarm_rows={beeswarm_rows}")

In [ ]:
# SHAP rows aligned to model rows (exposure as weight)
shap_tr, data_tr = sa.load_shap_data(output_base, "train")
shap_te, _ = sa.load_shap_data(output_base, "test", with_values=False)
n_rows = {"train": int(len(shap_tr)), "test": int(len(shap_te))}
print(f"SHAP rows - train: {n_rows['train']:,} | test: {n_rows['test']:,}")

In [ ]:
# Importance tables: model features and original columns (record level), train + test
enc_summary = pd.read_csv(f"{results_dir}/04c_encoding_summary.csv")
source_map = sa.feature_source_map(enc_summary)
enc_map = sa.encoding_type_map(enc_summary)

table = sa.importance_table(sa.shap_stats(shap_tr), sa.shap_stats(shap_te), source_map, enc_map)
col_shap_tr = sa.column_shap(shap_tr, source_map)
col_shap_te = sa.column_shap(shap_te, source_map)
by_col = sa.importance_by_column(sa.shap_stats(col_shap_tr), sa.shap_stats(col_shap_te), table, enc_map)
del shap_te, col_shap_te

table.to_csv(f"{results_dir}/07a_shap_importance.csv", index=False)
by_col.to_csv(f"{results_dir}/07a_shap_importance_by_column.csv", index=False)
save_parquet(col_shap_tr, f"{results_dir}/07a_column_shap_train.parquet")
print(f"Saved: 07a_shap_importance.csv ({len(table)} features), "
      f"07a_shap_importance_by_column.csv ({len(by_col)} columns), 07a_column_shap_train.parquet")

In [ ]:
# 1. Beeswarm by original column (train) - same order as the column importance chart below
col_order = by_col.loc[by_col["test_mean_abs_shap"] > 0, "source_column"].head(top_n).tolist()
onehot_cols = set(by_col.loc[by_col["n_encoded"] > 1, "source_column"])
charts = {}
charts["beeswarm_columns"] = sa.plot_beeswarm(
    col_shap_tr, data_tr, col_order, f"SHAP beeswarm - top {len(col_order)} columns (train)",
    f"{plots_dir}/07a_beeswarm_columns.png", max_rows=beeswarm_rows,
    grey_columns=[c for c in col_order if c in onehot_cols])
display(Image(filename=charts["beeswarm_columns"]))
print("Grey rows = one-hot columns (levels summed per record; no single feature value to colour by)")

In [ ]:
# 2. Key numbers
used = table[table["test_mean_abs_shap"] > 0]
zero = table.loc[table["test_mean_abs_shap"] == 0, "feature"].tolist()
coverage = sa.coverage_counts(table)
col_coverage = sa.coverage_counts(by_col)
top_te = set(table.head(top_n)["feature"])
top_tr = set(table.nsmallest(top_n, "train_rank")["feature"])
overlap = len(top_te & top_tr)
rank_corr = float(table.head(top_n)[["test_rank", "train_rank"]].astype(float).corr(method="spearman").iloc[0, 1])
col_top_te = set(by_col.head(top_n)["source_column"])
col_top_tr = set(by_col.nsmallest(top_n, "train_rank")["source_column"])
col_overlap = len(col_top_te & col_top_tr)

print(f"Model features:         {len(table)}  (used: {len(used)}, never used / SHAP = 0: {len(zero)})")
print(f"Original columns:       {len(by_col)}  (used: {int((by_col['test_mean_abs_shap'] > 0).sum())})")
print("Features to reach importance: " + ", ".join(f"{k.replace('pct', '%')}: {v}" for k, v in coverage.items()))
print("Columns to reach importance:  " + ", ".join(f"{k.replace('pct', '%')}: {v}" for k, v in col_coverage.items()))
print(f"Train vs test top-{top_n} features: overlap {overlap}/{top_n}, rank correlation {rank_corr:.3f}")
print(f"Train vs test top-{top_n} columns:  overlap {col_overlap}/{top_n}")

In [ ]:
# 3. Importance by original column (record level) - order used by the column beeswarm
charts["importance_columns"] = sa.plot_importance_bars(
    by_col, "test_pct", "source_column", top_n, f"SHAP importance - top {top_n} columns (test)",
    f"{plots_dir}/07a_importance_columns.png")
display(Image(filename=charts["importance_columns"]))
cols = ["source_column", "encoding_type", "n_encoded", "test_pct", "test_cum_pct", "test_rank",
        "train_rank", "test_pct_sum_of_levels"]
by_col[cols].head(top_n).round(2)

In [ ]:
# 4. Beeswarm by model feature (train) - same order as the feature importance chart below
feat_order = table.loc[table["test_mean_abs_shap"] > 0, "feature"].head(top_n).tolist()
charts["beeswarm_features"] = sa.plot_beeswarm(
    shap_tr, data_tr, feat_order, f"SHAP beeswarm - top {len(feat_order)} features (train)",
    f"{plots_dir}/07a_beeswarm_features.png", max_rows=beeswarm_rows)
display(Image(filename=charts["beeswarm_features"]))

In [ ]:
# 5. Importance by model feature
charts["importance_features"] = sa.plot_importance_bars(
    table, "test_pct", "feature", top_n, f"SHAP importance - top {top_n} features (test)",
    f"{plots_dir}/07a_importance_features.png")
display(Image(filename=charts["importance_features"]))
cols = ["feature", "source_column", "test_pct", "test_cum_pct", "test_rank", "train_rank", "rank_change"]
table[cols].head(top_n).round(2)

In [ ]:
# 6. Train vs test importance and cumulative importance (model features)
charts["train_vs_test"] = sa.plot_train_vs_test(table, top_n, f"{plots_dir}/07a_train_vs_test.png")
charts["cumulative"] = sa.plot_cumulative_importance(table, coverage, f"{plots_dir}/07a_cumulative.png")
for k in ("train_vs_test", "cumulative"):
    display(Image(filename=charts[k]))

In [ ]:
summary = {
    "timestamp": datetime.now().isoformat(),
    "rows": n_rows,
    "n_features": int(len(table)),
    "n_features_used": int(len(used)),
    "n_zero_shap": int(len(zero)),
    "zero_shap_features": zero,
    "n_columns": int(len(by_col)),
    "coverage_features": coverage,
    "coverage_columns": col_coverage,
    "train_test_top_overlap_features": int(overlap),
    "train_test_top_overlap_columns": int(col_overlap),
    "train_test_rank_corr_features": round(rank_corr, 4),
    "top_features": feat_order,
    "top_columns": col_order,
    "plots": {k: str(v) for k, v in charts.items()},
}
with open(f"{results_dir}/07a_summary.yaml", "w") as f:
    yaml.safe_dump(summary, f, sort_keys=False)
print(f"Saved: {results_dir}/07a_summary.yaml")

In [ ]:
print("\n########################################")
print("# STAGE 07a: COMPLETE")
print("########################################")